In [0]:
%sql
CREATE CATALOG IF NOT EXISTS ATMOS
  MANAGED LOCATION 'abfss://atmos-landing-dev-001@saatmosdevwestus2001.dfs.core.windows.net/';

CREATE SCHEMA   IF NOT EXISTS ATMOS.SILVER;

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, DateType

In [0]:
dbutils.widgets.text("ingestion_date", "", "Data de processamento (YYYY-MM-DD)")
ingestion_date = dbutils.widgets.get("ingestion_date")

In [0]:
df_bronze = (
    spark.table("atmos.bronze.inmet_microdados_raw")
    .filter(F.col("ingestion_date") == ingestion_date)
)

In [0]:
df_normalized = df_bronze.select(
    F.to_date(F.col("data")).cast(DateType())    .alias("data_observacao"),
    F.col("hora")                                .alias("hora_observacao"),
    F.col("id_estacao")                          .alias("id_estacao"),
    F.col("source_system")                       .alias("sistema_origem"),
    F.col("temperatura_bulbo_hora") .cast(DoubleType()).alias("temperatura_c"),
    F.col("temperatura_max")        .cast(DoubleType()).alias("temperatura_max_c"),
    F.col("temperatura_min")        .cast(DoubleType()).alias("temperatura_min_c"),
    F.col("temperatura_orvalho_hora").cast(DoubleType()).alias("ponto_orvalho_c"),
    F.col("umidade_rel_hora")       .cast(DoubleType()).alias("umidade_pct"),
    F.col("umidade_rel_max")        .cast(DoubleType()).alias("umidade_max_pct"),
    F.col("umidade_rel_min")        .cast(DoubleType()).alias("umidade_min_pct"),
    F.col("pressao_atm_hora")       .cast(DoubleType()).alias("pressao_hpa"),
    F.col("precipitacao_total")     .cast(DoubleType()).alias("precipitacao_mm"),
    F.col("radiacao_global")        .cast(DoubleType()).alias("radiacao_solar_wm2"),
    F.col("vento_velocidade")       .cast(DoubleType()).alias("velocidade_vento_ms"),
    F.col("vento_rajada_max")       .cast(DoubleType()).alias("rajada_vento_ms"),
    F.col("vento_direcao")          .cast(DoubleType()).alias("direcao_vento_graus"),
    F.lit(None)                     .cast(DoubleType()).alias("indice_uv"),
    F.to_date(F.lit(ingestion_date))             .alias("data_ingestao"),
)

In [0]:
df_estacoes = (
    spark.table("atmos.bronze.inmet_estacao_raw")
    .select(F.col("id_estacao"), F.col("estacao").alias("city"))
)

df_with_city = (
    df_normalized
    .join(df_estacoes, df_normalized["id_estacao"] == df_estacoes["id_estacao"], "left")
    .drop(df_estacoes["id_estacao"])
    .withColumnRenamed("city", "cidade")
)

In [0]:
df_clean = (
    df_with_city
    .filter(F.col("data_observacao").isNotNull())
    .filter(F.col("id_estacao").isNotNull())
    .withColumn("temperatura_c",
        F.when(F.col("temperatura_c").between(-20, 50), F.col("temperatura_c")))
    .withColumn("temperatura_max_c",
        F.when(F.col("temperatura_max_c").between(-20, 55), F.col("temperatura_max_c")))
    .withColumn("umidade_pct",
        F.when(F.col("umidade_pct").between(0, 100), F.col("umidade_pct")))
    .withColumn("pressao_hpa",
        F.when(F.col("pressao_hpa").between(870, 1050), F.col("pressao_hpa")))
    .withColumn("precipitacao_mm",
        F.when(F.col("precipitacao_mm") < 0, F.lit(0.0)).otherwise(F.col("precipitacao_mm")))
    .withColumn("velocidade_vento_ms",
        F.when(F.col("velocidade_vento_ms").between(0, 100), F.col("velocidade_vento_ms")))
    .withColumn("radiacao_solar_wm2",
        F.when(F.col("radiacao_solar_wm2").between(0, 4000), F.col("radiacao_solar_wm2")))
    .dropDuplicates(["data_observacao", "hora_observacao", "id_estacao"])
)

In [0]:
(
    df_clean.write
    .format("delta")
    .mode("overwrite")
    .option("replaceWhere", f"data_ingestao = '{ingestion_date}'")
    .partitionBy("data_ingestao")
    .saveAsTable("atmos.silver.climate_inmet")
)

In [0]:
count = (
    spark.table("atmos.silver.climate_inmet")
    .filter(F.col("data_ingestao") == ingestion_date)
    .count()
)

assert count > 0, f"Nenhum registro gravado em silver.climate_inmet para data_ingestao={ingestion_date}."
print(f"OK — {count} registros em silver.climate_inmet | data_ingestao={ingestion_date}")